In [1]:
import json
import os
from prettytable import PrettyTable


In [2]:
RESULTS_DIR = "../results/"

TASKS = ["spread", "lbf_hard"]
TASK_LABELS = {
    "spread": "Spread",
    "lbf_hard": "LBF ($\\times 10^2$)",
}
SEEDS = [42, 43, 44, 45, 46]

V7_CONFIGS = [
    "active_01",
    "active_02",
    "active_03",
    "free_01",
    "free_02",
]

REPORTED_V6 = {
    "spread": "active_03",
    "lbf_hard": "active_01",
}

NAME_MAP = {
    "pimac_v6": "PC3D{\\scriptsize-MAPPO}",
    "pimac_v7": "Hyper-PC3D{\\scriptsize-MAPPO}",
}

DISPLAY_SCALE = {
    "spread": 1.0,
    "lbf_hard": 100.0,
}


In [3]:
def get_path(task, algorithm, config, seed):
    dir_path = os.path.join(RESULTS_DIR, task, algorithm)
    files = os.listdir(dir_path)
    files = [f for f in files if f"s{seed}" in f and config in f]
    if len(files) == 0:
        raise ValueError(f"No run found for {task=} {algorithm=} {config=} {seed=}")
    if len(files) > 1:
        raise ValueError(f"Multiple runs found for {task=} {algorithm=} {config=} {seed=}: {files}")
    return os.path.join(dir_path, files[0])


def mean_std(values):
    mean = sum(values) / len(values)
    std = (sum((x - mean) ** 2 for x in values) / (len(values) - 1)) ** 0.5
    return mean, std


def format_score(mean, std):
    return f"${mean:.2f}$ $\\pm$ ${std:.2f}$"


def get_scores(task, algorithm, config):
    scores = {"train": [], "val": [], "test": []}
    for seed in SEEDS:
        path = get_path(task, algorithm, config, seed)
        with open(os.path.join(path, "summary.json"), "r") as f:
            data = json.load(f)
        scores["train"].append(data["test"]["train_counts_mean"] * DISPLAY_SCALE[task])
        scores["val"].append(data["test"]["validation_counts_mean"] * DISPLAY_SCALE[task])
        scores["test"].append(data["test"]["test_counts_mean"] * DISPLAY_SCALE[task])

    summary = {}
    for split, values in scores.items():
        summary[f"{split}_mean"], summary[f"{split}_std"] = mean_std(values)
        summary[split] = format_score(summary[f"{split}_mean"], summary[f"{split}_std"])

    summary["selection_score"] = min(summary["train_mean"], summary["val_mean"])
    return summary


In [4]:
v7_scores = {}
v7_rows = []

table = PrettyTable()
table.field_names = ["Task", "Config", "n", "Train", "Validation", "Test", "Selection"]

for task in TASKS:
    v7_scores[task] = {}
    for config in V7_CONFIGS:
        scores = get_scores(task, "pimac_v7", config)
        v7_scores[task][config] = scores
        v7_rows.append((task, config, scores))
        table.add_row([
            task,
            config,
            len(SEEDS),
            scores["train"],
            scores["val"],
            scores["test"],
            f"${scores['selection_score']:.2f}$",
        ])

print(table)


+----------+-----------+---+-----------------------+-----------------------+-----------------------+-----------+
|   Task   |   Config  | n |         Train         |       Validation      |          Test         | Selection |
+----------+-----------+---+-----------------------+-----------------------+-----------------------+-----------+
|  spread  | active_01 | 5 | $-40.45$ $\pm$ $0.90$ | $-49.16$ $\pm$ $1.14$ | $-81.45$ $\pm$ $2.37$ |  $-49.16$ |
|  spread  | active_02 | 5 | $-41.10$ $\pm$ $0.84$ | $-49.70$ $\pm$ $1.33$ | $-82.70$ $\pm$ $1.47$ |  $-49.70$ |
|  spread  | active_03 | 5 | $-40.54$ $\pm$ $0.78$ | $-48.96$ $\pm$ $0.99$ | $-80.73$ $\pm$ $2.27$ |  $-48.96$ |
|  spread  |  free_01  | 5 | $-40.87$ $\pm$ $0.85$ | $-49.38$ $\pm$ $1.46$ | $-81.44$ $\pm$ $2.56$ |  $-49.38$ |
|  spread  |  free_02  | 5 | $-39.62$ $\pm$ $1.00$ | $-48.41$ $\pm$ $1.40$ | $-80.58$ $\pm$ $1.95$ |  $-48.41$ |
| lbf_hard | active_01 | 5 |  $5.64$ $\pm$ $0.88$  |  $3.38$ $\pm$ $0.63$  |  $7.58$ $\pm$ $0.92

In [5]:
def make_v7_latex_table(rows):
    table_top = """
\\begin{table}[t]
\\centering
\\caption{Hyper-PC3D configuration-level final evaluation. LBF values are scaled by $10^2$.}
\\label{tab:v7_config_results}
\\begin{tabular}{@{}llcccc@{}}
\\toprule
Task & Config & Train & Validation & Test & Selection \\\\ \\midrule
"""
    table_bottom = """
\\bottomrule
\\end{tabular}
\\end{table}
"""
    body = []
    for task, config, scores in rows:
        task_name = "LBF ($\\times 10^2$)" if task == "lbf_hard" else "Spread"
        body.append(
            f"{task_name} & {config.replace('_', '\\_')} & {scores['train']} & {scores['val']} & {scores['test']} & ${scores['selection_score']:.2f}$ \\\\"
        )
    return table_top + "\n".join(body) + table_bottom


print(make_v7_latex_table(v7_rows))



\begin{table}[t]
\centering
\caption{Hyper-PC3D configuration-level final evaluation. LBF values are scaled by $10^2$.}
\label{tab:v7_config_results}
\begin{tabular}{@{}llcccc@{}}
\toprule
Task & Config & Train & Validation & Test & Selection \\ \midrule
Spread & active\_01 & $-40.45$ $\pm$ $0.90$ & $-49.16$ $\pm$ $1.14$ & $-81.45$ $\pm$ $2.37$ & $-49.16$ \\
Spread & active\_02 & $-41.10$ $\pm$ $0.84$ & $-49.70$ $\pm$ $1.33$ & $-82.70$ $\pm$ $1.47$ & $-49.70$ \\
Spread & active\_03 & $-40.54$ $\pm$ $0.78$ & $-48.96$ $\pm$ $0.99$ & $-80.73$ $\pm$ $2.27$ & $-48.96$ \\
Spread & free\_01 & $-40.87$ $\pm$ $0.85$ & $-49.38$ $\pm$ $1.46$ & $-81.44$ $\pm$ $2.56$ & $-49.38$ \\
Spread & free\_02 & $-39.62$ $\pm$ $1.00$ & $-48.41$ $\pm$ $1.40$ & $-80.58$ $\pm$ $1.95$ & $-48.41$ \\
LBF ($\times 10^2$) & active\_01 & $5.64$ $\pm$ $0.88$ & $3.38$ $\pm$ $0.63$ & $7.58$ $\pm$ $0.92$ & $3.38$ \\
LBF ($\times 10^2$) & active\_02 & $7.08$ $\pm$ $0.71$ & $3.89$ $\pm$ $0.48$ & $8.53$ $\pm$ $0.08$ & $3.89$

In [6]:
best_v7 = {}
for task in TASKS:
    best_v7[task] = max(V7_CONFIGS, key=lambda config: v7_scores[task][config]["selection_score"])

comparison = {}
for task in TASKS:
    comparison[task] = {
        "pimac_v6": {
            "config": REPORTED_V6[task],
            "scores": get_scores(task, "pimac_v6", REPORTED_V6[task]),
        },
        "pimac_v7": {
            "config": best_v7[task],
            "scores": get_scores(task, "pimac_v7", best_v7[task]),
        },
    }

table = PrettyTable()
table.field_names = ["Task", "Method", "Config", "Train", "Validation", "Test", "Selection"]
for task in TASKS:
    for algorithm in ["pimac_v6", "pimac_v7"]:
        entry = comparison[task][algorithm]
        scores = entry["scores"]
        table.add_row([
            task,
            NAME_MAP[algorithm],
            entry["config"],
            scores["train"],
            scores["val"],
            scores["test"],
            f"${scores['selection_score']:.2f}$",
        ])

print(table)
print(best_v7)


+----------+-------------------------------+-----------+-----------------------+-----------------------+-----------------------+-----------+
|   Task   |             Method            |   Config  |         Train         |       Validation      |          Test         | Selection |
+----------+-------------------------------+-----------+-----------------------+-----------------------+-----------------------+-----------+
|  spread  |    PC3D{\scriptsize-MAPPO}    | active_03 | $-39.90$ $\pm$ $0.72$ | $-48.09$ $\pm$ $1.03$ | $-79.18$ $\pm$ $1.52$ |  $-48.09$ |
|  spread  | Hyper-PC3D{\scriptsize-MAPPO} |  free_02  | $-39.62$ $\pm$ $1.00$ | $-48.41$ $\pm$ $1.40$ | $-80.58$ $\pm$ $1.95$ |  $-48.41$ |
| lbf_hard |    PC3D{\scriptsize-MAPPO}    | active_01 |  $7.91$ $\pm$ $0.70$  |  $4.47$ $\pm$ $0.30$  |  $8.98$ $\pm$ $0.08$  |   $4.47$  |
| lbf_hard | Hyper-PC3D{\scriptsize-MAPPO} | active_03 |  $6.73$ $\pm$ $1.02$  |  $3.97$ $\pm$ $0.61$  |  $8.54$ $\pm$ $0.63$  |   $3.97$  |
+----------+-

In [7]:
def make_comparison_latex_table(comparison):
    table_top = """
\\begin{table}[t]
\\centering
\\caption{Reported PC3D variant versus the best Hyper-PC3D configuration selected without test returns. LBF values are scaled by $10^2$.}
\\label{tab:v6_v7_comparison}
\\resizebox{\\textwidth}{!}{%
\\begin{tabular}{@{}lccc|ccc@{}}
\\toprule
\\multirow{2}{*}{Method} & \\multicolumn{3}{c}{Spread} & \\multicolumn{3}{c}{LBF ($\\times 10^2$)} \\\\ 
 & Train & Validation & Test & Train & Validation & Test \\\\ \\midrule
"""
    table_bottom = """
\\bottomrule
\\end{tabular}
}
\\end{table}
"""
    rows = []
    for algorithm in ["pimac_v6", "pimac_v7"]:
        row = [NAME_MAP[algorithm]]
        for task in TASKS:
            scores = comparison[task][algorithm]["scores"]
            row.extend([scores["train"], scores["val"], scores["test"]])
        rows.append(" & ".join(row) + " \\\\")
    return table_top + "\n".join(rows) + table_bottom


print(make_comparison_latex_table(comparison))



\begin{table}[t]
\centering
\caption{Reported PC3D variant versus the best Hyper-PC3D configuration selected without test returns. LBF values are scaled by $10^2$.}
\label{tab:v6_v7_comparison}
\resizebox{\textwidth}{!}{%
\begin{tabular}{@{}lccc|ccc@{}}
\toprule
\multirow{2}{*}{Method} & \multicolumn{3}{c}{Spread} & \multicolumn{3}{c}{LBF ($\times 10^2$)} \\ 
 & Train & Validation & Test & Train & Validation & Test \\ \midrule
PC3D{\scriptsize-MAPPO} & $-39.90$ $\pm$ $0.72$ & $-48.09$ $\pm$ $1.03$ & $-79.18$ $\pm$ $1.52$ & $7.91$ $\pm$ $0.70$ & $4.47$ $\pm$ $0.30$ & $8.98$ $\pm$ $0.08$ \\
Hyper-PC3D{\scriptsize-MAPPO} & $-39.62$ $\pm$ $1.00$ & $-48.41$ $\pm$ $1.40$ & $-80.58$ $\pm$ $1.95$ & $6.73$ $\pm$ $1.02$ & $3.97$ $\pm$ $0.61$ & $8.54$ $\pm$ $0.63$ \\
\bottomrule
\end{tabular}
}
\end{table}

